<a href="https://colab.research.google.com/github/gaelosvaldor-star/Simulaci-n-2-/blob/main/Caminata_aleatoria_grafo_Gael_Rosales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estudio de una caminata aleatoria sobre cadenas binarias

**Alumno:** Gael Osvaldo Rosales de la Cruz  
**Materia:** Simulación 2  
**Maestro:** Ricardo Medel Esquivel

---

## Propósito

Analizar las cadenas binarias de longitud `m` que no tienen dos unos consecutivos. El cuaderno calcula cuántas cadenas cumplen la condición, obtiene el promedio exacto de unos bajo una selección uniforme y lo compara con una simulación de Monte Carlo.

> Para ejecutarlo en Google Colab, abre Colab y selecciona **Archivo → Subir cuaderno**, luego elige este archivo `.ipynb`.


## 1. Modelo

Una cadena es **admisible** si cada elemento es 0 o 1 y no aparece el patrón `11`. Por ejemplo, `101001` es admisible y `101101` no lo es.

Representamos cada cadena admisible como un vértice. Dos vértices están conectados cuando una cadena se obtiene de la otra cambiando un solo bit y la nueva cadena sigue siendo admisible. La caminata elige una posición al azar y propone invertir su bit; si la inversión produciría dos unos vecinos, se queda en el mismo estado.

Cada cambio válido tiene probabilidad `1/m` en ambas direcciones. Por ello, la cadena de Markov es simétrica y su distribución estacionaria es uniforme sobre las cadenas admisibles. La tasa de aceptación solo describe qué proporción de propuestas cambia de estado; no es un valor teórico fijo como `1/φ`.


## 2. Conteo exacto y promedio teórico

Sea `A(m)` el número de cadenas admisibles de longitud `m`. Al separar las cadenas según si comienzan con 0 o con 10, se obtiene

`A(m) = A(m-1) + A(m-2)`, con `A(0)=1` y `A(1)=2`.

Para obtener el promedio de unos sin enumerar todas las cadenas, sea `S(m)` la suma del número de unos sobre todas las cadenas admisibles. La misma partición da

`S(m) = S(m-1) + S(m-2) + A(m-2)`.

Entonces el promedio exacto bajo la distribución uniforme es `S(m) / A(m)`. Estas recurrencias permiten manejar `m=100` sin construir los `2^m` estados.


In [ ]:
def conteo_y_promedio_exactos(m):
    """Devuelve (número de cadenas admisibles, promedio exacto de unos)."""
    if m < 0:
        raise ValueError("La longitud m debe ser un entero no negativo.")

    cantidad = [1, 2]  # A(0), A(1)
    suma_de_unos = [0, 1]  # S(0), S(1)

    for n in range(2, m + 1):
        cantidad.append(cantidad[n - 1] + cantidad[n - 2])
        suma_de_unos.append(
            suma_de_unos[n - 1] + suma_de_unos[n - 2] + cantidad[n - 2]
        )

    promedio = suma_de_unos[m] / cantidad[m]
    return cantidad[m], promedio


m = 100
cantidad_exacta, promedio_exacto = conteo_y_promedio_exactos(m)

print(f"Longitud de la cadena: {m}")
print(f"Cadenas admisibles: {cantidad_exacta:,}")
print(f"Promedio exacto de unos: {promedio_exacto:.6f}")

## 3. Simulación de Monte Carlo

La simulación comienza en la cadena de ceros. Durante el **calentamiento** (burn-in) no se guardan observaciones; después, se registra el número de unos de cada estado visitado. Para que el proceso sea reproducible se fija una semilla.

Como los estados consecutivos están correlacionados, el error estándar se aproxima con el método de **promedios por bloques**: se agrupan las observaciones y se calcula la variabilidad entre promedios de bloques. Es una aproximación cuyo resultado depende del tamaño de bloque y del tiempo de mezcla.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def simular_caminata(m, pasos, calentamiento, semilla=2026):
    """Simula la caminata y devuelve muestras, tasa de aceptación y estado final."""
    if m < 1:
        raise ValueError("m debe ser al menos 1.")
    if pasos <= calentamiento or calentamiento < 0:
        raise ValueError("Se requiere pasos > calentamiento >= 0.")

    rng = np.random.default_rng(semilla)
    estado = np.zeros(m, dtype=np.int8)
    unos = 0
    aceptados = 0
    muestras = np.empty(pasos - calentamiento, dtype=np.int16)
    indice_muestra = 0

    for t in range(pasos):
        posicion = int(rng.integers(m))

        if estado[posicion] == 1:
            estado[posicion] = 0
            unos -= 1
            aceptados += 1
        else:
            izquierda_libre = posicion == 0 or estado[posicion - 1] == 0
            derecha_libre = posicion == m - 1 or estado[posicion + 1] == 0
            if izquierda_libre and derecha_libre:
                estado[posicion] = 1
                unos += 1
                aceptados += 1

        if t >= calentamiento:
            muestras[indice_muestra] = unos
            indice_muestra += 1

    return muestras, aceptados / pasos, estado


pasos = 500_000
calentamiento = 100_000
semilla = 2026
muestras, tasa_aceptacion, estado_final = simular_caminata(
    m=m,
    pasos=pasos,
    calentamiento=calentamiento,
    semilla=semilla,
)

estimacion_mc = float(muestras.mean())


## 4. Resultados y precisión de la estimación

La comparación con el promedio exacto permite revisar la simulación. La tasa de aceptación se reporta como una característica observada de la caminata, no como una condición para que la media coincida.


In [ ]:
tamano_bloque = 1_000
numero_bloques = len(muestras) // tamano_bloque
promedios_bloque = muestras[:numero_bloques * tamano_bloque].reshape(
    numero_bloques, tamano_bloque
).mean(axis=1)
error_estandar_bloques = promedios_bloque.std(ddof=1) / np.sqrt(numero_bloques)

print(f"Longitud m: {m}")
print(f"Pasos simulados: {pasos:,} (calentamiento: {calentamiento:,})")
print(f"Semilla: {semilla}")
print(f"Tasa de aceptación observada: {tasa_aceptacion:.4f}")
print(f"Promedio estimado por Monte Carlo: {estimacion_mc:.5f}")
print(f"Error estándar aproximado por bloques: {error_estandar_bloques:.5f}")
print(f"Promedio exacto: {promedio_exacto:.5f}")
print(f"Diferencia absoluta: {abs(estimacion_mc - promedio_exacto):.5f}")
print(f"Estado final admisible: {not np.any(estado_final[:-1] + estado_final[1:] == 2)}")

## 5. Visualización

La primera gráfica muestra cómo cambia la media acumulada de las observaciones guardadas. La segunda permite observar la variación temporal y la distribución simulada del número de unos. La línea vertical de referencia corresponde al promedio exacto.


In [ ]:
medias_acumuladas = np.cumsum(muestras) / np.arange(1, len(muestras) + 1)
frecuencia_grafica = max(1, len(muestras) // 5_000)
indices = np.arange(1, len(muestras) + 1, frecuencia_grafica)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.8))

ejes[0].plot(indices, medias_acumuladas[::frecuencia_grafica], color="#1f77b4", lw=1)
ejes[0].axhline(promedio_exacto, color="#d62728", ls="--", label=f"Exacto = {promedio_exacto:.3f}")
ejes[0].set_title("Convergencia de la media acumulada")
ejes[0].set_xlabel("Observaciones posteriores al calentamiento")
ejes[0].set_ylabel("Media de unos")
ejes[0].grid(alpha=0.25)
ejes[0].legend()

# Distribución empírica del número de unos
ejes[1].hist(muestras, bins=np.arange(muestras.min() - 0.5, muestras.max() + 1.5),
             density=True, color="#66a182", edgecolor="white", alpha=0.9)
ejes[1].axvline(promedio_exacto, color="#d62728", ls="--", label=f"Exacto = {promedio_exacto:.3f}")
ejes[1].axvline(estimacion_mc, color="#ff7f0e", ls="-", label=f"Simulación = {estimacion_mc:.3f}")
ejes[1].set_title("Número de unos en los estados visitados")
ejes[1].set_xlabel("Cantidad de unos")
ejes[1].set_ylabel("Densidad")
ejes[1].grid(axis="y", alpha=0.25)
ejes[1].legend()

plt.tight_layout()
plt.show()

## 6. Interpretación

El número de cadenas admisibles se obtiene exactamente con una recurrencia, no mediante una aproximación de punto flotante. La caminata estima el promedio de unos y se contrasta contra un valor exacto calculado sin enumerar las cadenas.

Si la media simulada difiere ligeramente del promedio exacto, esa diferencia se debe al error de Monte Carlo y a la dependencia entre estados visitados. Aumentar el número de pasos o ajustar el calentamiento y el tamaño de bloque puede mejorar el análisis. La caminata usada es reversible porque cada cambio válido tiene la misma probabilidad en ambos sentidos.


